# Phase 15: Historical Walk-Forward Simulation — Replication Notebook



**Branch:** phase14/heterogeneous-ensemble  

**Phase context:** DJ-095 through DJ-110  

**Depends on:** `make walkforward-orchestrate` (or `make walkforward-smoke-full` for 22-ticker smoke test)



This notebook loads walk-forward ensemble results for a configurable **condition** and

**period**, then exercises the full MCP pipeline (compose → risk → allocate) and

generates the following analyses:



1. Ensemble output loading and validation

2. Buy/Hold/Sell decision distribution

3. Quality metrics: Shannon entropy and herding index

4. MCP pipeline execution (compose, risk, allocate)

5. Risk dashboard: VaR, drawdown, sector exposure

6. Return correlation structure

7. Capital allocation: order table, commission estimates

8. Portfolio composition: weights, sector breakdown

9. IC/IR: Spearman rank correlation with 21-day forward returns

10. Summary of key findings



**Re-runnable:** change `CONDITION` and `PERIOD` in the configuration cell to analyze any

completed condition.


In [ ]:
"""Setup: path resolution, imports, and configuration."""

import json

import sys

import warnings

from pathlib import Path



import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from scipy.stats import spearmanr



warnings.filterwarnings("ignore")



_nb = Path(".").resolve()

ROOT = _nb.parent if (_nb.parent / "src").exists() else _nb

sys.path.insert(0, str(ROOT / "src"))



# ── CONFIGURATION ── edit these cells to select condition / period ──────

CONDITION   = "full"           # full | parallel | homogeneous | no-memory

PERIOD      = "held-out-test"  # held-out-test | validation | walk-forward

DATA_DIR    = ROOT / "data"

OUTPUT_DIR  = DATA_DIR / "walkforward"

CAPITAL     = 500_000.0

# ────────────────────────────────────────────────────────────────────────



print(f"ROOT       : {ROOT}")

print(f"CONDITION  : {CONDITION}")

print(f"PERIOD     : {PERIOD}")

print(f"OUTPUT_DIR : {OUTPUT_DIR}")

print(f"CAPITAL    : ${CAPITAL:,.0f}")


---

## 1. Load Ensemble Outputs



Reads all `{ticker}.json` files from `OUTPUT_DIR/{CONDITION}/{YYYY}/{MM}/`.  

Each file is the `EnsembleOutput` produced by `aggregate_agent_outputs()` in

`src/hifi/simulation/agent_executor.py`.  

If the directory is empty, run `make walkforward-orchestrate` (or

`make walkforward-smoke-full` for the 22-ticker smoke test) first.


In [ ]:
from hifi.data.universe import get_sector

from hifi.simulation.schedule import get_period_dates



dates = get_period_dates(PERIOD)

cond_dir = OUTPUT_DIR / CONDITION

records = []

missing_dates = []



for date in dates:

    year, month, _ = date.split("-")

    date_dir = cond_dir / year / month

    if not date_dir.exists():

        missing_dates.append(date)

        continue

    for path in sorted(date_dir.glob("*.json")):

        if path.stem == "portfolio":

            continue

        try:

            data = json.loads(path.read_text(encoding="utf-8"))

            ed = data.get("ensemble_decision", {})

            records.append({

                "date":       date,

                "ticker":     path.stem,

                "decision":   ed.get("collective_decision") or "Hold",

                "confidence": float(ed.get("collective_confidence") or 0.0),

                "sector":     get_sector(path.stem) or "Unknown",

            })

        except Exception as exc:

            print(f"WARN {path.name}: {exc}")



df = pd.DataFrame(records)

n_dates  = df["date"].nunique()  if not df.empty else 0

n_tickers = df["ticker"].nunique() if not df.empty else 0



print(f"Period: {PERIOD} — {len(dates)} expected dates")

print(f"Loaded: {len(df):,} records  |  {n_dates} dates  |  {n_tickers} tickers")

print(f"Missing dates: {len(missing_dates)}")

if df.empty:

    print("\nNO DATA — run the orchestrator first:")

    print("  make walkforward-orchestrate")

    print("  (or make walkforward-smoke-full for the 22-ticker smoke test)")

else:

    print("\nDecision counts:")

    print(df["decision"].value_counts().to_string())


---

## 2. Decision Distribution



Buy/Hold/Sell distribution overall and broken down by GICS sector.

A well-calibrated ensemble should show Buy rates commensurate with the prevailing

market regime: high-Buy in bull periods, high-Hold/Sell in bear periods.


In [ ]:
if df.empty:

    print("PENDING — no data loaded")

else:

    COLORS = {"Buy": "#2ecc71", "Hold": "#95a5a6", "Sell": "#e74c3c"}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))



    # Overall distribution

    counts = df["decision"].value_counts()

    axes[0].bar(

        counts.index, counts.values,

        color=[COLORS.get(d, "gray") for d in counts.index],

    )

    axes[0].set_title(f"Decision Distribution\n{CONDITION} / {PERIOD}")

    axes[0].set_ylabel("Count")

    for i, (label, val) in enumerate(counts.items()):

        axes[0].text(i, val + 0.5, str(val), ha="center", fontsize=10)



    # By sector

    pivot = df.groupby(["sector", "decision"]).size().unstack(fill_value=0)

    for col in ["Buy", "Hold", "Sell"]:

        if col not in pivot.columns:

            pivot[col] = 0

    pivot[["Buy", "Hold", "Sell"]].plot(

        kind="bar", ax=axes[1], stacked=True,

        color=["#2ecc71", "#95a5a6", "#e74c3c"],

    )

    axes[1].set_title("Decisions by GICS Sector")

    axes[1].set_xlabel("")

    axes[1].legend(title="Decision", bbox_to_anchor=(1.05, 1))

    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=35, ha="right")

    plt.tight_layout()

    plt.savefig(DATA_DIR / f"phase15_decisions_{CONDITION}.png", dpi=120, bbox_inches="tight")

    plt.show()



    n_total = len(df)

    print(f"Buy:  {counts.get('Buy',  0):4d}  ({counts.get('Buy',  0)/n_total:.1%})")

    print(f"Hold: {counts.get('Hold', 0):4d}  ({counts.get('Hold', 0)/n_total:.1%})")

    print(f"Sell: {counts.get('Sell', 0):4d}  ({counts.get('Sell', 0)/n_total:.1%})")


In [ ]:
"""Confidence heatmap: ticker × date, colour = confidence level."""

if df.empty:

    print("PENDING")

else:

    pivot_conf = df.pivot_table(

        index="ticker", columns="date", values="confidence", aggfunc="mean"

    )

    fig, ax = plt.subplots(

        figsize=(max(12, len(pivot_conf.columns) * 0.6),

                 max(6, len(pivot_conf) * 0.35))

    )

    im = ax.imshow(pivot_conf.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)

    ax.set_xticks(range(len(pivot_conf.columns)))

    ax.set_xticklabels(pivot_conf.columns, rotation=45, ha="right", fontsize=7)

    ax.set_yticks(range(len(pivot_conf.index)))

    ax.set_yticklabels(pivot_conf.index, fontsize=8)

    ax.set_title(f"Confidence Heatmap — {CONDITION} / {PERIOD}")

    plt.colorbar(im, ax=ax, label="Collective Confidence")

    plt.tight_layout()

    plt.show()

    print(f"Mean confidence: {df['confidence'].mean():.3f}  Std: {df['confidence'].std():.3f}")


---

## 3. Quality Metrics: Entropy and Herding



**Shannon entropy** (normalized to [0,1]) measures decision diversity per date.  

Entropy = 1.0 means equal Buy/Hold/Sell counts (maximum diversity).  

Entropy = 0.0 means unanimous agreement (complete herding).  



**Herding index** = fraction of tickers that match the majority decision.  

From complexity science: herding > 0.8 indicates the ensemble is operating

in a low-entropy regime where the collective decision is dominated by a single

viewpoint, reducing the epistemic value of multi-agent deliberation.



Phase 14 E0 baseline (5-org ensemble, 98 tickers, 2022-Q3):  

mean_entropy=0.7449, mean_herding=0.7667


In [ ]:
from scipy.stats import entropy as scipy_entropy



if df.empty:

    print("PENDING")

else:

    ent_rows  = []

    herd_rows = []

    MAX_H = np.log2(3)  # max entropy for 3 outcomes



    for date, grp in df.groupby("date"):

        probs = grp["decision"].value_counts(normalize=True).values

        h = float(scipy_entropy(probs, base=2))

        ent_rows.append({"date": date, "entropy": h,

                          "normalized_entropy": h / MAX_H if MAX_H > 0 else 0})

        n = len(grp)

        maj = grp["decision"].value_counts().iloc[0]

        herd_rows.append({"date": date, "herding": maj / n})



    ent_df  = pd.DataFrame(ent_rows)

    herd_df = pd.DataFrame(herd_rows)



    fig, axes = plt.subplots(1, 2, figsize=(14, 4))



    axes[0].plot(ent_df["date"], ent_df["normalized_entropy"], "o-", color="navy")

    axes[0].axhline(0.5, color="gray", linestyle="--", alpha=0.6, label="50% entropy")

    axes[0].set_title("Normalized Shannon Entropy by Date")

    axes[0].set_ylim(0, 1.05)

    axes[0].set_ylabel("Entropy (normalized)")

    axes[0].legend()

    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha="right")



    axes[1].plot(herd_df["date"], herd_df["herding"], "s-", color="darkred")

    axes[1].axhline(0.80, color="orange", linestyle="--", alpha=0.8, label="80% threshold")

    axes[1].set_title("Herding Index by Date")

    axes[1].set_ylim(0, 1.05)

    axes[1].set_ylabel("Majority fraction")

    axes[1].legend()

    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")



    plt.tight_layout()

    plt.savefig(DATA_DIR / f"phase15_entropy_herding_{CONDITION}.png", dpi=120, bbox_inches="tight")

    plt.show()



    print(f"Mean entropy (normalized) : {ent_df['normalized_entropy'].mean():.3f}")

    print(f"Mean herding index        : {herd_df['herding'].mean():.3f}")

    print(f"Phase 14 E0 baseline      : entropy=0.7449  herding=0.7667")


---

## 4. MCP Pipeline Execution



Applies the three MCP tools programmatically to the most recent date with available data:



1. `compose_portfolio(signals)` — converts decisions to target weights (max 5% per stock, 20% per sector)

2. `compute_risk_report(portfolio, ohlcv)` — VaR 95/99, drawdown, sector cap, correlation annotation

3. `generate_orders(approved_weights, prices, holdings, capital)` — IBKR-commission-aware orders



This is the same chain implemented in `src/hifi/simulation/pipeline.py`.


In [ ]:
from hifi.simulation.pipeline import run_pipeline



if df.empty:

    print("PENDING — no ensemble data")

    snapshot = None

    latest_date = None

    ohlcv = {}

else:

    latest_date = df["date"].max()

    latest_df   = df[df["date"] == latest_date].copy()

    print(f"Running MCP pipeline on {latest_date} ({len(latest_df)} tickers) ...")



    # Load OHLCV (last 90 trading days up to date)

    ohlcv: dict = {}

    for ticker in latest_df["ticker"].unique():

        path = DATA_DIR / "market" / ticker / "ohlcv.parquet"

        if path.exists():

            tdf = pd.read_parquet(path)

            tdf.index = pd.to_datetime(tdf.index)

            window = tdf[tdf.index <= latest_date].tail(90)

            if not window.empty:

                ohlcv[ticker] = [

                    {"date": str(idx.date()), "close": float(row["close"])}

                    for idx, row in window.iterrows()

                ]



    prices = {t: rows[-1]["close"] for t, rows in ohlcv.items() if rows}

    signals = latest_df.to_dict("records")



    portfolio_state = {

        "portfolio":       {},

        "portfolio_value": CAPITAL,

        "hwm_value":       CAPITAL,

        "holdings":        {},

        "prices":          prices,

    }

    constraints = {

        "max_single_stock": 0.05,

        "max_sector":       0.20,

        "min_position":     0.005,

        "capital":          CAPITAL,

        "current_capital":  0.0,

    }



    snapshot = run_pipeline(signals, ohlcv, portfolio_state, constraints)

    rr = snapshot.risk_report



    print(f"Pipeline complete:")

    print(f"  Buy/Hold/Sell    : {snapshot.n_buy} / {snapshot.n_hold} / {snapshot.n_sell}")

    print(f"  Orders generated : {len(snapshot.orders)}")

    print(f"  Total notional   : ${snapshot.total_estimated_value:,.0f}")

    print(f"  Approved signals : {len(rr.get('approved_signals', []))}")

    print(f"  Blocked tickers  : {len(rr.get('blocked_tickers', []))}")

    var_95 = rr.get('portfolio_var_95', 0)

    var_99 = rr.get('portfolio_var_99', 0)

    print(f"  Portfolio VaR 95%: {var_95:.2%}  VaR 99%: {var_99:.2%}")


---

## 5. Risk Dashboard



Sector exposure (must stay below 20% cap) and portfolio-level risk metrics.

Red bars indicate sectors at or near the concentration cap.


In [ ]:
if snapshot is None:

    print("PENDING")

else:

    rr = snapshot.risk_report

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))



    # Sector exposure

    if snapshot.sector_exposure:

        sectors = list(snapshot.sector_exposure.keys())

        exps    = [snapshot.sector_exposure[s] for s in sectors]

        bar_colors = ["#e74c3c" if e >= 0.195 else "#3498db" for e in exps]

        axes[0].barh(sectors, exps, color=bar_colors)

        axes[0].axvline(0.20, color="red", linestyle="--", linewidth=1.5, label="20% cap")

        axes[0].set_title(f"Sector Exposure — {latest_date}")

        axes[0].set_xlabel("Portfolio Weight")

        axes[0].legend()

        for i, (s, e) in enumerate(zip(sectors, exps)):

            axes[0].text(e + 0.002, i, f"{e:.1%}", va="center", fontsize=9)



    # Risk metrics bar chart

    var_95 = abs(rr.get("portfolio_var_95", 0))

    var_99 = abs(rr.get("portfolio_var_99", 0))

    drawdown = abs(rr.get("current_drawdown", 0))

    risk_labels = ["VaR 95%", "VaR 99%", "Drawdown"]

    risk_values = [var_95 * 100, var_99 * 100, drawdown * 100]

    axes[1].bar(risk_labels, risk_values, color=["coral", "tomato", "firebrick"])

    axes[1].set_title("Risk Metrics (% of Portfolio)")

    axes[1].set_ylabel("Percent")

    for i, v in enumerate(risk_values):

        axes[1].text(i, v + 0.05, f"{v:.2f}%", ha="center", fontsize=10)



    plt.tight_layout()

    plt.savefig(DATA_DIR / f"phase15_risk_{CONDITION}.png", dpi=120, bbox_inches="tight")

    plt.show()



    print(f"VaR 95%: {var_95:.2%}   VaR 99%: {var_99:.2%}   Drawdown: {drawdown:.2%}")

    blocked = rr.get("blocked_tickers", [])

    if blocked:

        print(f"Blocked tickers: {blocked}")


---

## 6. Return Correlation Structure



Pearson correlation matrix of daily returns over the 90-day OHLCV window.  

The risk manager uses this matrix to annotate highly correlated position pairs

(correlation > 0.85 threshold triggers a warning but does not block).  

From a portfolio construction perspective, low-correlation Buy signals provide

more diversification value than high-correlation clusters.


In [ ]:
if not ohlcv:

    print("PENDING — no OHLCV data")

else:

    prices_s = pd.DataFrame({

        t: pd.Series({r["date"]: r["close"] for r in rows})

        for t, rows in ohlcv.items() if rows

    }).sort_index()

    returns_s = prices_s.pct_change().dropna()



    if returns_s.shape[1] < 2 or len(returns_s) < 20:

        print("Insufficient data for correlation matrix")

    else:

        corr = returns_s.corr()

        n_tickers_c = len(corr)

        fig, ax = plt.subplots(

            figsize=(min(16, n_tickers_c * 0.55), min(14, n_tickers_c * 0.55))

        )

        im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)

        ax.set_xticks(range(n_tickers_c))

        ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)

        ax.set_yticks(range(n_tickers_c))

        ax.set_yticklabels(corr.index, fontsize=7)

        ax.set_title(f"Return Correlation Matrix ({len(returns_s)} days ending {latest_date})")

        plt.colorbar(im, ax=ax, label="Pearson Correlation")

        plt.tight_layout()

        plt.savefig(DATA_DIR / f"phase15_corr_{CONDITION}.png", dpi=120, bbox_inches="tight")

        plt.show()



        # Top 5 most positively correlated Buy-signal pairs

        buy_tickers = set(df[df["decision"] == "Buy"]["ticker"].unique())

        print("Top 5 highest correlations among available tickers:")

        flat = corr.unstack()

        flat = flat[flat.index.get_level_values(0) < flat.index.get_level_values(1)]

        top = flat.abs().nlargest(5)

        for (t1, t2), _ in top.items():

            r = corr.loc[t1, t2]

            buy_flag = " ← both Buy" if t1 in buy_tickers and t2 in buy_tickers else ""

            print(f"  {t1}-{t2}: {r:+.3f}{buy_flag}")


---

## 7. Capital Allocation: Orders and Commissions



The capital allocator (`src/hifi/mcp/capital_allocator.py`) converts approved weights

to integer share quantities using IBKR tiered commissions:

- Shares ≤ 300: $0.0035/share (minimum $0.35)

- Shares > 300: $0.0020/share (capped at 1% of trade value)

- Kelly criterion cap applied before position sizing


In [ ]:
if snapshot is None or not snapshot.orders:

    print("PENDING or no orders generated (all Buy signals may have been blocked)")

else:

    orders_df = pd.DataFrame(snapshot.orders)

    print(f"Orders for {latest_date}: {len(orders_df)} positions")

    print()



    # Format for display

    disp = orders_df.copy()

    if "notional" in disp:

        disp["notional"] = disp["notional"].apply(lambda x: f"${x:,.0f}")

    if "commission" in disp:

        disp["commission"] = disp["commission"].apply(lambda x: f"${x:.2f}")

    if "price" in disp:

        disp["price"] = disp["price"].apply(lambda x: f"${x:.2f}")



    show_cols = [c for c in ["ticker", "action", "quantity", "price", "notional", "commission"]

                 if c in disp.columns]

    print(disp[show_cols].to_string(index=False))



    if "commission" in orders_df.columns and "notional" in orders_df.columns:

        total_comm = orders_df["commission"].sum()

        total_not  = orders_df["notional"].sum()

        cash_remaining = CAPITAL - total_not

        print(f"\nTotal notional   : ${total_not:,.0f}")

        print(f"Total commission : ${total_comm:.2f}  ({total_comm/total_not:.3%} of notional)")

        print(f"Cash remaining   : ${cash_remaining:,.0f}  ({cash_remaining/CAPITAL:.1%})")


---

## 8. Portfolio Composition



Final position weights and sector allocation after applying the MCP pipeline.  

The portfolio must satisfy: no single stock > 5%, no sector > 20%.


In [ ]:
if snapshot is None:

    print("PENDING")

else:

    # Derive weights from order notionals

    weights: dict = {}

    if snapshot.orders:

        total_not = sum(o.get("notional", 0) for o in snapshot.orders)

        if total_not > 0:

            weights = {o["ticker"]: o.get("notional", 0) / CAPITAL

                       for o in snapshot.orders}



    if not weights:

        print("No invested positions — all Buy signals blocked or no Buy signals")

    else:

        sector_weights: dict = {}

        for ticker, w in weights.items():

            s = get_sector(ticker) or "Unknown"

            sector_weights[s] = sector_weights.get(s, 0) + w

        cash_w = max(0.0, 1.0 - sum(weights.values()))

        if cash_w > 0.001:

            sector_weights["Cash"] = cash_w



        fig, axes = plt.subplots(1, 2, figsize=(14, max(6, len(weights) * 0.35)))



        # Weight bar chart (horizontal)

        w_sorted = dict(sorted(weights.items(), key=lambda x: -x[1]))

        axes[0].barh(list(w_sorted.keys()), list(w_sorted.values()), color="#3498db")

        axes[0].axvline(0.05, color="red", linestyle="--", linewidth=1.2, label="5% max")

        axes[0].set_title(f"Position Weights — {latest_date}")

        axes[0].set_xlabel("Weight")

        axes[0].legend()



        # Sector pie

        axes[1].pie(

            list(sector_weights.values()),

            labels=list(sector_weights.keys()),

            autopct="%1.1f%%",

            startangle=90,

        )

        axes[1].set_title("Sector Allocation")



        plt.tight_layout()

        plt.savefig(DATA_DIR / f"phase15_portfolio_{CONDITION}.png", dpi=120, bbox_inches="tight")

        plt.show()



        invested = sum(weights.values())

        print(f"Positions: {len(weights)}  Invested: {invested:.1%}  Cash: {1-invested:.1%}")


---

## 9. IC / IR: Predictive Signal Quality



**Information Coefficient (IC)** = Spearman rank correlation between:

- Predicted score: Buy=+1, Hold=0, Sell=-1

- Realized 21-day forward return



**Information Ratio (IR)** = mean_IC / std_IC  



IC > 0 means the ensemble correctly ranked higher-return assets with higher Buy conviction.  

IC requires forward return data (not available until 21 days after the evaluation date).  

For the held-out test period (2022-2023), forward returns are fully available by 2024.



For complete IC/IR computation across all conditions use:  

`make walkforward-ic` (runs `scripts/compute_phase15_ic.py`).


In [ ]:
DECISION_SCORE = {"Buy": 1.0, "Hold": 0.0, "Sell": -1.0}



if df.empty:

    print("PENDING — no ensemble data")

else:

    ic_records: list = []



    for date, grp in df.groupby("date"):

        scores = grp["decision"].map(DECISION_SCORE).fillna(0).values

        fwd_returns: list = []



        for _, row in grp.iterrows():

            path = DATA_DIR / "market" / row["ticker"] / "ohlcv.parquet"

            if path.exists():

                try:

                    tdf = pd.read_parquet(path)

                    tdf.index = pd.to_datetime(tdf.index)

                    future = tdf[tdf.index > date]["close"]

                    fwd_returns.append(

                        float(future.iloc[21] / future.iloc[0] - 1)

                        if len(future) >= 22 else float("nan")

                    )

                except Exception:

                    fwd_returns.append(float("nan"))

            else:

                fwd_returns.append(float("nan"))



        fwd = np.array(fwd_returns)

        valid = ~np.isnan(fwd)

        if valid.sum() >= 5:

            ic_val, p_val = spearmanr(scores[valid], fwd[valid])

            ic_records.append({"date": date, "ic": ic_val, "p": p_val, "n": int(valid.sum())})



    if not ic_records:

        print("IC computation: forward return data not yet available.")

        print("This is expected if the walk-forward period has not fully elapsed.")

        print("Run: make walkforward-ic  (after 2024-01-01 for held-out-test period)")

    else:

        ic_df    = pd.DataFrame(ic_records)

        mean_ic  = float(ic_df["ic"].mean())

        std_ic   = float(ic_df["ic"].std())

        ir       = mean_ic / std_ic if std_ic > 0 else float("nan")

        n_pos    = int((ic_df["ic"] > 0).sum())



        fig, ax = plt.subplots(figsize=(12, 4))

        bar_colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in ic_df["ic"]]

        ax.bar(range(len(ic_df)), ic_df["ic"].values, color=bar_colors)

        ax.set_xticks(range(len(ic_df)))

        ax.set_xticklabels(ic_df["date"], rotation=45, ha="right", fontsize=8)

        ax.axhline(0, color="black", linewidth=0.8)

        ax.axhline(mean_ic, color="navy", linestyle="--",

                   label=f"Mean IC = {mean_ic:.4f}")

        ax.set_title(f"Information Coefficient by Date — {CONDITION}")

        ax.set_ylabel("Spearman IC")

        ax.legend()

        plt.tight_layout()

        plt.savefig(DATA_DIR / f"phase15_ic_{CONDITION}.png", dpi=120, bbox_inches="tight")

        plt.show()



        print(f"Mean IC : {mean_ic:.4f}")

        print(f"IC Std  : {std_ic:.4f}")

        print(f"IR      : {ir:.4f}")

        print(f"Positive IC dates: {n_pos}/{len(ic_df)}")

        print(f"Mean p-value      : {ic_df['p'].mean():.4f}")


---

## 10. Summary



Key findings from this replication run.  

Copy this section into the capstone report or bitacora.


In [ ]:
SEP = "=" * 70

print(SEP)

print("PHASE 15 WALK-FORWARD — REPLICATION SUMMARY")

print(SEP)



if df.empty:

    print("\nNO DATA — run the orchestrator first:")

    print("  make walkforward-orchestrate")

else:

    n_total = len(df)

    counts  = df["decision"].value_counts()

    print(f"\nCondition : {CONDITION}")

    print(f"Period    : {PERIOD}")

    print(f"Dates     : {df['date'].nunique()}  |  Tickers: {df['ticker'].nunique()}")

    print(f"Records   : {n_total:,}")

    print(f"\nDecision distribution:")

    for dec in ["Buy", "Hold", "Sell"]:

        n = counts.get(dec, 0)

        print(f"  {dec}: {n:5d}  ({n/n_total:.1%})")

    print(f"\nMean confidence  : {df['confidence'].mean():.3f}")

    if 'ent_df' in dir():

        print(f"Mean entropy     : {ent_df['normalized_entropy'].mean():.3f}")

        print(f"Mean herding     : {herd_df['herding'].mean():.3f}")

    if snapshot is not None:

        rr = snapshot.risk_report

        print(f"\nPipeline ({latest_date}):")

        print(f"  Orders         : {len(snapshot.orders)}")

        print(f"  Total notional : ${snapshot.total_estimated_value:,.0f}")

        print(f"  VaR 95%%        : {rr.get('portfolio_var_95', 0):.2%}")

        print(f"  Blocked        : {len(rr.get('blocked_tickers', []))} tickers")

    if 'ic_records' in dir() and ic_records:

        print(f"\nIC / IR:")

        print(f"  Mean IC : {mean_ic:.4f}")

        print(f"  IR      : {ir:.4f}")

        print(f"  n_dates : {len(ic_df)}")



print(f"\n{SEP}")
